In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

import wfdb
import ast

import matplotlib.pyplot as plt
import seaborn as sns

## 1. Read Data

In [2]:
curr_path = Path.cwd()
dataset_folder = curr_path.parent.parent / 'Datasets' / 'ptb_ecg_dataset'

In [3]:
ptbxl_path = dataset_folder / 'ptbxl_database.csv'
ptbxl_df = pd.read_csv(ptbxl_path, index_col='ecg_id')

In [4]:
ptbxl_df.head()

,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,...,SLI-LVH,QRS-CLBB,ST-ELEV-MI,ST-DEPR-MI,Q-ISC,Q-ISC-QPeak,Q-ISC-V2V3,Q-ISC-RPeak,STRAIN,MI-ALL
ecg_id,,,,,,,,,,,,,,,,,,,,,
1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,sinusrhythmus periphere niederspannung,...,False,False,False,False,False,False,False,False,False,False
2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,sinusbradykardie sonst normales ekg,...,True,False,False,False,False,False,False,False,False,False
3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,sinusrhythmus normales ekg,...,False,False,False,False,False,False,False,False,False,False
4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,sinusrhythmus normales ekg,...,False,False,False,False,False,False,False,False,False,False
5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,sinusrhythmus normales ekg,...,False,False,False,False,False,False,False,False,False,False


In [5]:
ptbxl_df.scp_codes

ecg_id
1                 {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}
2                             {'NORM': 80.0, 'SBRAD': 0.0}
3                               {'NORM': 100.0, 'SR': 0.0}
4                               {'NORM': 100.0, 'SR': 0.0}
5                               {'NORM': 100.0, 'SR': 0.0}
                               ...                        
21833    {'NDT': 100.0, 'PVC': 100.0, 'VCLVH': 0.0, 'ST...
21834             {'NORM': 100.0, 'ABQRS': 0.0, 'SR': 0.0}
21835                           {'ISCAS': 50.0, 'SR': 0.0}
21836                           {'NORM': 100.0, 'SR': 0.0}
21837                           {'NORM': 100.0, 'SR': 0.0}
Name: scp_codes, Length: 21801, dtype: object

In [6]:
ptbxl_df.shape

(21801, 43)

In [7]:
ptbxl_df.isnull().sum()

patient_id                          0
age                                 0
sex                                 0
height                          14826
weight                          12379
nurse                            1475
site                               18
device                              0
recording_date                      0
report                              0
scp_codes                           0
heart_axis                       8470
infarction_stadium1             16188
infarction_stadium2             21698
validated_by                     9380
second_opinion                      0
initial_autogenerated_report        0
validated_by_human                  0
baseline_drift                  20202
static_noise                    18541
burst_noise                     21188
electrodes_problems             21771
extra_beats                     19852
pacemaker                       21510
strat_fold                          0
filename_lr                         0
filename_hr 

In [8]:
ptbxl_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21801 entries, 1 to 21837
Data columns (total 43 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   patient_id                    21801 non-null  float64
 1   age                           21801 non-null  float64
 2   sex                           21801 non-null  int64  
 3   height                        6975 non-null   float64
 4   weight                        9422 non-null   float64
 5   nurse                         20326 non-null  float64
 6   site                          21783 non-null  float64
 7   device                        21801 non-null  object 
 8   recording_date                21801 non-null  object 
 9   report                        21801 non-null  object 
 10  scp_codes                     21801 non-null  object 
 11  heart_axis                    13331 non-null  object 
 12  infarction_stadium1           5613 non-null   object 
 13  infarc

In [9]:
ptbxl_df.describe()

,patient_id,age,sex,height,weight,nurse,site,validated_by,strat_fold
count,21801.000000,21801.000000,21801.000000,6975.000000,9422.000000,20326.000000,21783.000000,12421.000000,21801.000000
mean,11250.554287,62.769781,0.479106,166.703226,70.996391,2.291745,1.545012,0.746075,5.503142
std,6235.025560,32.307421,0.499575,10.866804,15.878365,3.254033,4.172799,1.178003,2.874868
min,302.000000,2.000000,0.000000,6.000000,5.000000,0.000000,0.000000,0.000000,1.000000
25%,5975.000000,50.000000,0.000000,160.000000,60.000000,0.000000,0.000000,0.000000,3.000000
50%,11419.000000,62.000000,0.000000,166.000000,70.000000,1.000000,1.000000,1.000000,6.000000
75%,16608.000000,72.000000,1.000000,174.000000,80.000000,3.000000,2.000000,1.000000,8.000000
max,21797.000000,300.000000,1.000000,209.000000,250.000000,11.000000,50.000000,11.000000,10.000000


In [10]:
ptbxl_df['nurse']

ecg_id
1        2.0
2        2.0
3        2.0
4        2.0
5        2.0
        ... 
21833    1.0
21834    1.0
21835    1.0
21836    1.0
21837    1.0
Name: nurse, Length: 21801, dtype: float64

In [11]:
ptbxl_df['scp_codes']

ecg_id
1                 {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}
2                             {'NORM': 80.0, 'SBRAD': 0.0}
3                               {'NORM': 100.0, 'SR': 0.0}
4                               {'NORM': 100.0, 'SR': 0.0}
5                               {'NORM': 100.0, 'SR': 0.0}
                               ...                        
21833    {'NDT': 100.0, 'PVC': 100.0, 'VCLVH': 0.0, 'ST...
21834             {'NORM': 100.0, 'ABQRS': 0.0, 'SR': 0.0}
21835                           {'ISCAS': 50.0, 'SR': 0.0}
21836                           {'NORM': 100.0, 'SR': 0.0}
21837                           {'NORM': 100.0, 'SR': 0.0}
Name: scp_codes, Length: 21801, dtype: object

## Read the SCP Statements.csv

In [12]:
scp_path = dataset_folder / 'scp_statements.csv'
scp_df = pd.read_csv(scp_path)

In [13]:
scp_df.head()

,Unnamed: 0,description,diagnostic,form,rhythm,diagnostic_class,diagnostic_subclass,Statement Category,SCP-ECG Statement Description,AHA code,aECG REFID,CDISC Code,DICOM Code
0,NDT,non-diagnostic T abnormalities,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,non-diagnostic T abnormalities,NaN,NaN,NaN,NaN
1,NST_,non-specific ST changes,1.0,1.0,NaN,STTC,NST_,Basic roots for coding ST-T changes and abnorm...,non-specific ST changes,145.0,MDC_ECG_RHY_STHILOST,NaN,NaN
2,DIG,digitalis-effect,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,suggests digitalis-effect,205.0,NaN,NaN,NaN
3,LNGQT,long QT-interval,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,long QT-interval,148.0,NaN,NaN,NaN
4,NORM,normal ECG,1.0,NaN,NaN,NORM,NORM,Normal/abnormal,normal ECG,1.0,NaN,NaN,F-000B7


In [14]:
scp_df.shape

(71, 13)

In [15]:
ptbxl_df['filename_lr'].head()

ecg_id
1    records100/00000/00001_lr
2    records100/00000/00002_lr
3    records100/00000/00003_lr
4    records100/00000/00004_lr
5    records100/00000/00005_lr
Name: filename_lr, dtype: object

In [16]:
ptbxl_df.columns.tolist()

['patient_id',
 'age',
 'sex',
 'height',
 'weight',
 'nurse',
 'site',
 'device',
 'recording_date',
 'report',
 'scp_codes',
 'heart_axis',
 'infarction_stadium1',
 'infarction_stadium2',
 'validated_by',
 'second_opinion',
 'initial_autogenerated_report',
 'validated_by_human',
 'baseline_drift',
 'static_noise',
 'burst_noise',
 'electrodes_problems',
 'extra_beats',
 'pacemaker',
 'strat_fold',
 'filename_lr',
 'filename_hr',
 'r_peaks',
 'RS-LVH',
 'S12-LVH',
 'R56-LVH',
 'QRS-LVH',
 'LI-LVH',
 'SLI-LVH',
 'QRS-CLBB',
 'ST-ELEV-MI',
 'ST-DEPR-MI',
 'Q-ISC',
 'Q-ISC-QPeak',
 'Q-ISC-V2V3',
 'Q-ISC-RPeak',
 'STRAIN',
 'MI-ALL']

In [17]:
ptbxl_df.strat_fold

ecg_id
1        3
2        2
3        5
4        3
5        4
        ..
21833    7
21834    4
21835    2
21836    8
21837    9
Name: strat_fold, Length: 21801, dtype: int64

In [18]:
scp_df.columns.tolist()

['Unnamed: 0',
 'description',
 'diagnostic',
 'form',
 'rhythm',
 'diagnostic_class',
 'diagnostic_subclass',
 'Statement Category',
 'SCP-ECG Statement Description',
 'AHA code',
 'aECG REFID',
 'CDISC Code',
 'DICOM Code']

## 2. Reading raw single data

In [19]:
sampling_rate=100

In [26]:
Output_Datasets_path = curr_path.parent.parent / 'Output_Datasets'
wfdb_array_path = Output_Datasets_path / 'wfdb_array.npy'

In [27]:
if wfdb_array_path.exists():
    print('Already exits no need to read the wfdb data\nDirectly load the data array')
    
else:
    if sampling_rate == 100:
        data = [wfdb.rdsamp(fr'{dataset_folder}/'+f) for f in ptbxl_df.filename_lr]
    else:
        data = [wfdb.rdsamp(dataset_folder+f) for f in ptbxl_df.filename_hr]
    data = np.array([signal for signal, meta in data])

Already exits no need to read the wfdb data
Directly load the data array


In [30]:
if wfdb_array_path.exists():
    print('Directly read the data array')
else:
    print('Saving the wfbd_array...')
    np.save(fr'{data_array_path}\wfdb_array', data)
    print('Saved data array successfully')

Directly read the data array


In [31]:
# load the data array
print('Loading data_array.npy ...')
data = np.load(wfdb_array_path)
print('Loading successful')

Loading data_array.npy ...
Loading successful


In [32]:
print(type(data))
print(f'shape of whole data {data.shape}')

<class 'numpy.ndarray'>
shape of whole data (21801, 1000, 12)


In [33]:
data.size, data.size * data.itemsize * 1e-9

(261612000, 2.092896)

### 2.1. Extract signals

In [34]:
print(data[0])

print(f'Shape of a single unit: {data[0].shape}')

[[-0.119 -0.055  0.064 ... -0.026 -0.039 -0.079]
 [-0.116 -0.051  0.065 ... -0.031 -0.034 -0.074]
 [-0.12  -0.044  0.076 ... -0.028 -0.029 -0.069]
 ...
 [ 0.069  0.    -0.069 ...  0.024 -0.041 -0.058]
 [ 0.086  0.004 -0.081 ...  0.242 -0.046 -0.098]
 [ 0.022 -0.031 -0.054 ...  0.143 -0.035 -0.12 ]]
Shape of a single unit: (1000, 12)


### 2.2. Create Diagnositc Labels

In [35]:
# parse scp_codes from sting to dic
ptbxl_df['scp_codes'] = ptbxl_df['scp_codes'].apply(lambda x: ast.literal_eval(x))

In [36]:
ptbxl_df['scp_codes']

ecg_id
1                 {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}
2                             {'NORM': 80.0, 'SBRAD': 0.0}
3                               {'NORM': 100.0, 'SR': 0.0}
4                               {'NORM': 100.0, 'SR': 0.0}
5                               {'NORM': 100.0, 'SR': 0.0}
                               ...                        
21833    {'NDT': 100.0, 'PVC': 100.0, 'VCLVH': 0.0, 'ST...
21834             {'NORM': 100.0, 'ABQRS': 0.0, 'SR': 0.0}
21835                           {'ISCAS': 50.0, 'SR': 0.0}
21836                           {'NORM': 100.0, 'SR': 0.0}
21837                           {'NORM': 100.0, 'SR': 0.0}
Name: scp_codes, Length: 21801, dtype: object

In [37]:
scp_df.columns

Index(['Unnamed: 0', 'description', 'diagnostic', 'form', 'rhythm',
       'diagnostic_class', 'diagnostic_subclass', 'Statement Category',
       'SCP-ECG Statement Description', 'AHA code', 'aECG REFID', 'CDISC Code',
       'DICOM Code'],
      dtype='object')

In [38]:
2**32/1.25e-10

3.4359738367999996e+19

In [48]:
scp_df['diagnostic'].sum(), scp_df['diagnostic'].shape

(44.0, (71,))

In [49]:
# load label definitions & keep only diagnostic codes
diagnostic_df = scp_df[scp_df['diagnostic'] == 1].set_index('Unnamed: 0')

In [50]:
diagnostic_df.shape

(44, 12)

In [51]:
diagnostic_df

,description,diagnostic,form,rhythm,diagnostic_class,diagnostic_subclass,Statement Category,SCP-ECG Statement Description,AHA code,aECG REFID,CDISC Code,DICOM Code
Unnamed: 0,,,,,,,,,,,,
NDT,non-diagnostic T abnormalities,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,non-diagnostic T abnormalities,NaN,NaN,NaN,NaN
NST_,non-specific ST changes,1.0,1.0,NaN,STTC,NST_,Basic roots for coding ST-T changes and abnorm...,non-specific ST changes,145.0,MDC_ECG_RHY_STHILOST,NaN,NaN
DIG,digitalis-effect,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,suggests digitalis-effect,205.0,NaN,NaN,NaN
LNGQT,long QT-interval,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,long QT-interval,148.0,NaN,NaN,NaN
NORM,normal ECG,1.0,NaN,NaN,NORM,NORM,Normal/abnormal,normal ECG,1.0,NaN,NaN,F-000B7
IMI,inferior myocardial infarction,1.0,NaN,NaN,MI,IMI,Myocardial Infarction,inferior myocardial infarction,161.0,NaN,NaN,NaN
ASMI,anteroseptal myocardial infarction,1.0,NaN,NaN,MI,AMI,Myocardial Infarction,anteroseptal myocardial infarction,165.0,NaN,NaN,NaN
LVH,left ventricular hypertrophy,1.0,NaN,NaN,HYP,LVH,Ventricular Hypertrophy,left ventricular hypertrophy,142.0,NaN,C71076,NaN
LAFB,left anterior fascicular block,1.0,NaN,NaN,CD,LAFB/LPFB,Intraventricular and intra-atrial Conduction d...,left anterior fascicular block,101.0,MDC_ECG_BEAT_BLK_ANT_L_HEMI,C62267,D3-33140


In [52]:
diagnostic_df.loc['NDT', 'diagnostic_class']

'STTC'

### 2.3. Map SCP Codes → 5 Diagnostic Superclasses

In [54]:
def aggregate_diagnostic(scp_dict):
    """
    Maps a patient's raw SCP codes to diagnostic superclasses.
    
    Input:  {'IMI': 80.0, 'ISCA': 20.0}  (from ptbxl_df.scp_codes)
    Output: ['MI', 'STTC']                (superclass labels)
    """
    result = []
    for code in scp_dict.keys():
        if code in diagnostic_df.index:
            result.append(diagnostic_df.loc[code, 'diagnostic_class'])
    return list(set(result))  # set() removes duplicates

# Apply to every row
ptbxl_df['diagnostic_superclass'] = ptbxl_df.scp_codes.apply(aggregate_diagnostic)

ptbxl_df[['scp_codes', 'diagnostic_superclass']].head(10)

,scp_codes,diagnostic_superclass
ecg_id,,
1,"{'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}",[NORM]
2,"{'NORM': 80.0, 'SBRAD': 0.0}",[NORM]
3,"{'NORM': 100.0, 'SR': 0.0}",[NORM]
4,"{'NORM': 100.0, 'SR': 0.0}",[NORM]
5,"{'NORM': 100.0, 'SR': 0.0}",[NORM]
6,"{'NORM': 100.0, 'SR': 0.0}",[NORM]
7,"{'NORM': 100.0, 'SR': 0.0}",[NORM]
8,"{'IMI': 35.0, 'ABQRS': 0.0, 'SR': 0.0}",[MI]
9,"{'NORM': 100.0, 'SR': 0.0}",[NORM]


### 2.4. Train / Validation / Test Split (using strat_fold)

In [55]:
# Split signals (X) — folds 1-8 train, 9 val, 10 test
X_train = data[np.where(ptbxl_df.strat_fold <= 8)]
X_val   = data[np.where(ptbxl_df.strat_fold == 9)]
X_test  = data[np.where(ptbxl_df.strat_fold == 10)]

# Split labels (y)
y_train = ptbxl_df[ptbxl_df.strat_fold <= 8].diagnostic_superclass
y_val   = ptbxl_df[ptbxl_df.strat_fold == 9].diagnostic_superclass
y_test  = ptbxl_df[ptbxl_df.strat_fold == 10].diagnostic_superclass

print(f'Train: {X_train.shape[0]} records | shape: {X_train.shape}')
print(f'Val:   {X_val.shape[0]} records  | shape: {X_val.shape}')
print(f'Test:  {X_test.shape[0]} records  | shape: {X_test.shape}')

Train: 17420 records | shape: (17420, 1000, 12)
Val:   2183 records  | shape: (2183, 1000, 12)
Test:  2198 records  | shape: (2198, 1000, 12)


In [ ]:
np.save(rf'{Output_Datasets_path}/X_train', X_train)
np.save(rf'{Output_Datasets_path}/X_val', X_val)
np.save(rf'{Output_Datasets_path}/X_test', X_test)

In [58]:
np.save(rf'{Output_Datasets_path}/Y_train', y_train)
np.save(rf'{Output_Datasets_path}/Y_val', y_val)
np.save(rf'{Output_Datasets_path}/Y_test', y_test)

### 2.5. Convert Multi-Labels to Binary Vectors

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
y_train_bin = mlb.fit_transform(y_train)
y_val_bin   = mlb.transform(y_val)
y_test_bin  = mlb.transform(y_test)

print(f'Classes: {mlb.classes_}')
print(f'y_train shape: {y_train_bin.shape}')
print(f'y_val shape:   {y_val_bin.shape}')
print(f'y_test shape:  {y_test_bin.shape}')

# Example: first training record
print(f'\nFirst record labels: {y_train.iloc[0]}')
print(f'As binary vector:   {y_train_bin[0]}')

### 2.6. Filter Out Empty Labels

In [ ]:
# Some records have no diagnostic superclass (rhythm/form only)
# These need to be filtered out before training

mask_train = y_train.apply(len) > 0
mask_val   = y_val.apply(len) > 0
mask_test  = y_test.apply(len) > 0

X_train_clean = X_train[mask_train.values]
y_train_clean = y_train_bin[mask_train.values]

X_val_clean = X_val[mask_val.values]
y_val_clean = y_val_bin[mask_val.values]

X_test_clean = X_test[mask_test.values]
y_test_clean = y_test_bin[mask_test.values]

print(f'After filtering empty labels:')
print(f'  Train: {X_train.shape[0]} -> {X_train_clean.shape[0]} records')
print(f'  Val:   {X_val.shape[0]} -> {X_val_clean.shape[0]} records')
print(f'  Test:  {X_test.shape[0]} -> {X_test_clean.shape[0]} records')